# Embeddings and Language Model


---




*   Embeddings
*   Language models





## What Is An Embedding?

An embedding is a numerical representation (a vector or list of numbers) of complex data like words, images, or audio, that captures its meaning and relationships, allowing machine learning models to process it efficiently, with similar items placed closer together in a low-dimensional space.

# Tokenization

A tokenizer| is a tool in Natural Language Processing (NLP) that breaks down raw text into smaller, manageable units called tokens (words, subwords, or characters) and converts them into numerical IDs that machine learning models can process, making text understandable for models.

https://gpt-tokenizer.dev/   
GPT's tokenizer is trained with a Byte-Pair Encoding (BPE) algorithm

In [1]:
from transformers import T5Tokenizer

tokenizer = T5Tokenizer.from_pretrained("t5-base")
tokenized_sequence = tokenizer.tokenize("i am a student")
print(tokenized_sequence)
ids = tokenizer.convert_tokens_to_ids(tokenized_sequence)
print(ids)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


['▁', 'i', '▁am', '▁', 'a', '▁student']
[3, 23, 183, 3, 9, 1236]


## Embeddings
For demonstration purposes, Ww download a distilled GPT-2 model. It has 81.9M parameters.

An embedding layer converts the token 'ids' to a high-dimension vector.  It is a linear layer.  

In distilgpt2, the size of the embedding layer  is Embedding(50257, 768), meaning that the vocabulary size is 50257, and the dimension is 768



In [2]:
import torch
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained('distilgpt2')

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [3]:
total_params = sum(p.numel() for p in model.parameters())
print(f" → {total_params / 1e6:.1f}M parameters")

 → 81.9M parameters


We can extract the embedding layer from this model by

In [5]:
embedding_layer = model.transformer.wte

We can then use it to convert our low-dimensional input 'ids' to an high-dimensional input. Before, we need to first convert the 'ids' from a List object to a pytorch tensor:

In [6]:
tensor_ids = torch.tensor(ids)
print('Tensor ids:', tensor_ids)
print('Shape of the tensor: ', tensor_ids.shape)

Tensor ids: tensor([   3,   23,  183,    3,    9, 1236])
Shape of the tensor:  torch.Size([6])


Then we can use the embedding layer to convert the low-dimensional tensor to the high-dimensional representation

In [7]:
token_embedding = embedding_layer(tensor_ids)
print('Embedding of the tensor ids: \n', token_embedding)
print('Shape of the embedding: ', token_embedding.shape)
print(embedding_layer)

Embedding of the tensor ids: 
 tensor([[-0.0893, -0.2819,  0.1948,  ...,  0.1721,  0.0493, -0.1354],
        [-0.0834, -0.0459,  0.1357,  ...,  0.0459,  0.1461, -0.0952],
        [ 0.0294, -0.0517,  0.0921,  ...,  0.0827, -0.0021,  0.0955],
        [-0.0893, -0.2819,  0.1948,  ...,  0.1721,  0.0493, -0.1354],
        [-0.0646,  0.0156,  0.1146,  ...,  0.0321, -0.0473,  0.0548],
        [-0.2595, -0.0392,  0.0737,  ..., -0.0005, -0.2206,  0.1508]],
       grad_fn=<EmbeddingBackward0>)
Shape of the embedding:  torch.Size([6, 768])
Embedding(50257, 768)


# Language Models

## Preparing the dataset

In [ ]:
# !pip install datasets

For each of those tasks, we will use the [Wikitext 2](https://huggingface.co/datasets/Salesforce/wikitext
) dataset as an example. You can load it very easily with the 🤗 Datasets library.

In [8]:
from datasets import load_dataset
import nltk
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.lm import MLE

nltk.download('punkt', quiet=True)

# Load & clean
dataset = load_dataset('wikitext', 'wikitext-2-raw-v1')
train_texts = dataset['train']['text']
# partial
train_texts = train_texts[:1000]
train_texts = [line.strip() for line in train_texts if line.strip() and not line.startswith(" =")]

print(f"Loaded {len(train_texts):,} cleaned lines")

N = 2

# Prepare data
train_data, padded_sents = padded_everygram_pipeline(N, train_texts)


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Loaded 472 cleaned lines


In [10]:
import textwrap

print(textwrap.fill(dataset["train"][5]['text'], width=70))

 It met with positive sales in Japan , and was praised by both
Japanese and western critics . After release , it received
downloadable content , along with an expanded edition in November of
that year . It was also adapted into manga and an original video
animation series . Due to low sales of Valkyria Chronicles II ,
Valkyria Chronicles III was not localized , but a fan translation
compatible with the game 's expanded edition was released in 2014 .
Media.Vision would return to the franchise with the development of
Valkyria : Azure Revolution for the PlayStation 4 .


In [11]:
# Train
model = MLE(N)
model.fit(train_data, padded_sents)

In [12]:

print(f"Vocabulary size: {len(model.vocab):,}")
print(f"Number of {N}-grams: {model.counts[N].N():,}")


Vocabulary size: 156
Number of 2-grams: 281,626


In [13]:
print("\nWith prompt 'what's your name':")
print(" ".join(model.generate(35, text_seed=["It", "met"], random_seed=123)))


With prompt 'what's your name':
  .   . </s>   f   t   a l a   c a p a l e s   5 4   t h e r o l o f i n


# Pre-trained language model

---



In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
model = AutoModelForCausalLM.from_pretrained("distilgpt2")


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [16]:
prompt = "What's ai?"

inputs = tokenizer(prompt, return_tensors="pt")  # this gives input_ids + attention_mask

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    top_p=0.92,
    temperature=0.85,
    pad_token_id=tokenizer.eos_token_id,          # ← suppresses the second warning
    eos_token_id=tokenizer.eos_token_id,
)

print(textwrap.fill(tokenizer.decode(outputs[0], skip_special_tokens=True)))

What's ai?    I've always been a fan of H&M, but I've never even
looked at a single video. When I watched a movie, I thought it might
be an excellent idea to look at it, and then I saw that I could really
get the sense that this was the perfect moment for me. I started
thinking about H&M in the beginning. I think it could be an excellent
idea for me. So, I started to look at it as a way to


In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

I am Qwen, an AI language model created by Alibaba Cloud. I was designed to assist users with their questions and provide information on various topics. My primary function is to engage in natural language conversations


# https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct?library=transformers
# google search huggingface, model name:Qwen/Qwen2.5-0.5B-Instruct.  Then click 'use this model'
# sentence: How are you doing?
# print out tokens id
# print out embedding vectors for those tokens

In [18]:
model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [19]:
dir(model)

['T_destination',
 '__annotations__',
 '__call__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_adjust_missing_and_unexpected_keys',
 '_apply',
 '_assisted_decoding',
 '_auto_class',
 '_backward_compatibility_gradient_checkpointing',
 '_backward_hooks',
 '_backward_pre_hooks',
 '_beam_search',
 '_beam_search_has_unfinished_sequences',
 '_buffers',
 '_cache_dependant_input_preparation',
 '_cache_dependant_input_preparation_exporting',
 '_call_impl',
 '_can_compile_fullgraph',
 '_can_record_outputs',
 '_can_set_attn_implementation',
 '_check_and_adjust_attn_implementation',
 '_check_early_stop_heuristic',
 '_checkpoi

In [ ]:
model.get_input_embeddings()

Embedding(151936, 896)

In [31]:
tokenizer('what are you doing?')


{'input_ids': [10919, 389, 345, 1804, 30], 'attention_mask': [1, 1, 1, 1, 1]}

In [25]:
model.get_input_embeddings().weight

Parameter containing:
tensor([[-0.1445, -0.0455,  0.0042,  ..., -0.1523,  0.0184,  0.0991],
        [ 0.0573, -0.0722,  0.0234,  ...,  0.0603, -0.0042,  0.0478],
        [-0.1106,  0.0386,  0.1948,  ...,  0.0421, -0.1141, -0.1455],
        ...,
        [-0.0710, -0.0173,  0.0176,  ...,  0.0834,  0.1340, -0.0746],
        [ 0.1993,  0.0201,  0.0151,  ..., -0.0829,  0.0750, -0.0294],
        [ 0.0342,  0.0640,  0.0305,  ...,  0.0291,  0.0942,  0.0639]],
       requires_grad=True)

In [27]:
model.get_input_embeddings().weight[[10919, 389, 345, 1804, 30]]

tensor([[ 0.0029, -0.0829,  0.1194,  ...,  0.3203,  0.0664, -0.0452],
        [ 0.0956,  0.0556,  0.0474,  ...,  0.1087, -0.0230,  0.1331],
        [-0.0836,  0.1017,  0.0184,  ..., -0.1854, -0.1129, -0.0670],
        [-0.0035, -0.1503,  0.0591,  ..., -0.2028, -0.0401,  0.0922],
        [-0.1052, -0.0564,  0.0291,  ..., -0.1166,  0.1062, -0.1773]],
       grad_fn=<IndexBackward0>)

In [23]:
len(model.get_input_embeddings().weight[14623])

768

#exercise

In [ ]:
#convert a random sentence into tokens
#extract the embedding layer and print its size